In [65]:
import pandas as pd
import numpy as np
import os, sys
import itertools
import glob
import re
import difflib
import datetime

In [66]:
notebook_dir = os.path.dirname(os.getcwd())
source_data_path=os.path.join(notebook_dir, "Common Source Data")
sys.path.append(source_data_path)
from country_codes import countries
from country_regions import regions
from country_regions import region_iso3
from country_regions import iso3_region


In [67]:
reference_disease_iso3s_years=pd.read_csv(os.path.join(notebook_dir,'Imputation','df_impute_skeleton.csv')).drop(columns=['Unnamed: 0'])
diseases=[i for i in reference_disease_iso3s_years.columns if ('incidence' in i ) | ('vaccine' in i)]
poss_diseases=np.unique([i.split('_')[0] for i in diseases])
poss_ISO3s=np.unique(reference_disease_iso3s_years['ISO3']).tolist()+['FLK'] #FLK reports prohibition information, so may include as well for other diease absence info
poss_years=[int(i) for i in np.unique(reference_disease_iso3s_years['Year'])]


In [68]:
wahis_reports_poultry= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','poultry_admin_div_reports.csv'))
wahis_reports_cattle= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','cattle_admin_div_reports.csv'))
wahis_reports_pigs= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','swine_admin_div_reports.csv'))

wahis_reports_tot=pd.concat([wahis_reports_poultry,wahis_reports_cattle,wahis_reports_pigs])

wahis_reports_tot['Disease'] = wahis_reports_tot['Disease'].replace({
    'Newcastle disease virus (Inf. with)': 'Newcastle disease (velogenic)'
})

# Duplicate velogenic rows and relabel as "Newcastle disease", then append to main df

mask = wahis_reports_tot['Disease'].eq('Newcastle disease (velogenic)')
to_dup = wahis_reports_tot.loc[mask].copy()
to_dup['Disease'] = 'Newcastle disease'

# If 'Disease' is categorical, make sure the new label exists
if pd.api.types.is_categorical_dtype(wahis_reports_tot['Disease']):
    wahis_reports_tot['Disease'] = wahis_reports_tot['Disease'].cat.add_categories(['Newcastle disease'])

# Append the duplicated rows back into the main dataframe
wahis_reports_tot = pd.concat([wahis_reports_tot, to_dup], ignore_index=True)

In [69]:
wahis_reported_cases = wahis_reports_tot.drop_duplicates(subset=["Year", "Country", "Disease"]).reset_index(drop=True).loc[:,['Year','Country','Disease']]
wahis_reported_cases['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in wahis_reported_cases['Country']]
wahis_reported_cases['Status']='Present'

In [70]:
WAHIS_disease_sit=pd.read_csv(os.path.join(source_data_path, 'WAHIS data','Disease situation.csv'))

In [71]:
WAHIS_disease_sit.loc[WAHIS_disease_sit['Occurence code'] == 'Disease limited to one or more zones', 'Disease status'] = 'Present'
WAHIS_disease_sit.loc[WAHIS_disease_sit['Disease'] == 'Newcastle disease virus (Inf. with)', 'Disease']='Newcastle disease (velogenic)'
# Duplicate velogenic rows and relabel as "Newcastle disease", then append to main df

#Do not want to duplicate announcements of 'absence' for ND, as nonvelogenic is far more pervasive and not tracked.
mask = (
    WAHIS_disease_sit['Disease'].eq('Newcastle disease (velogenic)') &
    WAHIS_disease_sit['Disease status'].eq('Present')
)

to_dup = WAHIS_disease_sit.loc[mask].copy()
to_dup['Disease'] = 'Newcastle disease'

if pd.api.types.is_categorical_dtype(WAHIS_disease_sit['Disease']):
    WAHIS_disease_sit['Disease'] = WAHIS_disease_sit['Disease'].cat.add_categories(['Newcastle disease'])

WAHIS_disease_sit = pd.concat([WAHIS_disease_sit, to_dup], ignore_index=True)

WAHIS_disease_sit.rename(columns={'Disease status':'Status'},inplace=True)

In [72]:
WAHIS_disease_sit['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in WAHIS_disease_sit['Country']]

#Two cities listed as countries are dropped here
WAHIS_disease_sit = WAHIS_disease_sit[~WAHIS_disease_sit['Country'].isin(['Ceuta', 'Melilla'])]

In [73]:
df = WAHIS_disease_sit.copy()

group_cols = ['ISO3', 'Disease', 'Year','Animal category']
if 'Disease' not in df.columns and 'Animal category' in df.columns:
    group_cols = ['ISO3', 'Animal category', 'Year']

priority = {'Present': 3, 'Suspected': 2, 'Absent': 1}
df['_score'] = df['Status'].map(priority).fillna(0)

best_idx = df.groupby(group_cols)['_score'].idxmax()

resolved_WAHIS_dis_sit = df.loc[best_idx].copy()

# If a group had no Present/Suspected/Absent, set status to "Unknown"
resolved_WAHIS_dis_sit.loc[resolved_WAHIS_dis_sit['_score'] == 0, 'Status'] = 'Unknown'

resolved_WAHIS_dis_sit = resolved_WAHIS_dis_sit.drop(columns=['_score']).reset_index(drop=True)
resolved_WAHIS_dis_sit_domestic=resolved_WAHIS_dis_sit[resolved_WAHIS_dis_sit['Animal category']=="Domestic"]
resolved_WAHIS_dis_sit_wild=resolved_WAHIS_dis_sit[resolved_WAHIS_dis_sit['Animal category']=="Wild"]

In [74]:
folder_path =  os.path.join(source_data_path, "FAO Data","EMPRES-i+_reports")

csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

df_list = [pd.read_csv(file, encoding="utf-8") for file in csv_files]
fao_reports_df = pd.concat(df_list, ignore_index=True)
fao_reports_df=fao_reports_df[fao_reports_df['Diagnosis.status']!='Denied']


#Sometimes, when NICD reports are filed, the FAO report data is shifted to the right in the raw datafiles (correcting):
cols = list(fao_reports_df.columns)
j = cols.index("Diagnosis.status")
mask = fao_reports_df["Diagnosis.status"].eq(" NICD")
fao_reports_df.loc[mask, cols[j:-1]] = fao_reports_df.loc[mask, cols[j+1:]].to_numpy()

fao_reports_df['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in fao_reports_df['Country']]


In [75]:
fao_reports_df['Year']=[int(i.split('/')[2]) if (i==i and i!=' ') else int(j.split('/')[2]) for i,j in 
 zip(fao_reports_df['Observation.date..dd.mm.yyyy.'],fao_reports_df['Report.date..dd.mm.yyyy.']) ]

In [76]:
fao_reports_df = fao_reports_df.sort_values(by=["Disease", "ISO3", "Year"]).drop_duplicates(subset=["Disease", "ISO3","Year","Animal.type"], keep="first")

In [77]:
#Accounting for diseases that do not match very well with fuzzy matching
FAO_dict_replace=dict()
FAO_dict_replace['Bluetongue']='Bluetongue virus (Inf. with)'
FAO_dict_replace['Lumpy skin disease']='Lumpy skin disease virus (Inf. with)'
FAO_dict_replace['Peste des petits ruminants']='Peste des petits ruminants virus (Inf. with)'
FAO_dict_replace['Rabies']='Rabies virus (Inf. with)'
FAO_dict_replace['Rift Valley fever']='Rift Valley fever virus (Inf. with)'
FAO_dict_replace['Foot and mouth disease']='Foot and mouth disease virus (Inf. with)'
FAO_dict_replace['Trichinellosis']='Trichinella spp. (Inf. with)'
FAO_dict_replace['Leishmaniosis']='Leishmania spp. (Inf. with) (Leishmaniosis)'
FAO_dict_replace['Infectious bovine rhinotracheitis']='Infectious bovine rhinotracheitis/infectious pustular vulvovaginitis'
FAO_dict_replace['Echinococcosis/hydatidosis']='Echinococcus multilocularis (Inf. with) (2014-)'
FAO_dict_replace['Contagious equine metritis']='Taylorella equigenitalis (Contagious equine metritis) (Inf. with)'
FAO_dict_replace['Avian mycoplasmosis (M. gallisepticum)']='Mycoplasma gallisepticum\xa0(Avian mycoplasmosis) (Inf. with)'
FAO_dict_replace['Newcastle disease']=['Newcastle disease (velogenic)','Newcastle disease']

In [78]:
#Fuzzy matching FAO/disease free report disease names to match WAHIS conventions
def clean_name(s: str) -> str:
    """Lowercase; remove parenthetical garbage & punctuation."""
    if s is None:
        return ""
    s = str(s).casefold()
    s = re.sub(r"\([^)]*\)", " ", s)        
    s = s.replace("-", " ")
    s = re.sub(r"[/_,.]", " ", s)
    s = re.sub(r"\b(virus|viruses|disease|infection|infections|inf|with)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def best_match_one(target, choices_clean, choices_raw, scorer="auto"):
    """Return score for target."""
    t = clean_name(target)
    if not t or not choices_raw:
        return None, 0.0

    else:
        matches = difflib.get_close_matches(t, choices_clean, n=1, cutoff=0.0)
        if not matches:
            return None, 0.0
        best = matches[0]
        score = difflib.SequenceMatcher(a=t, b=best).ratio() * 100.0
        idx = choices_clean.index(best)
        return choices_raw[idx], score

def map_poss_to_df(
    df_new: pd.DataFrame,
    poss_diseases: list,
    threshold: float = 78.0,
    one_to_one: bool = False,
):
    """Map each poss_diseases item to its best Disease"""
    fao_unique_raw = sorted(set(df_new["Disease"].astype(str)))
    fao_unique_clean = [clean_name(x) for x in fao_unique_raw]

    used_idx = set()
    results = []
    for poss in poss_diseases:
        best_raw, score = best_match_one(poss, fao_unique_clean, fao_unique_raw)
        matched = None
        if best_raw is not None and score >= threshold:
            if one_to_one:
                # If already used, pick next best
                tmp_clean = list(fao_unique_clean)
                tmp_raw = list(fao_unique_raw)
                while True:
                    idx = tmp_raw.index(best_raw)
                    if idx not in used_idx:
                        used_idx.add(idx)
                        matched = best_raw
                        break
                    tmp_clean.pop(idx); tmp_raw.pop(idx)
                    best_raw, score = best_match_one(poss, tmp_clean, tmp_raw)
                    if best_raw is None or score < threshold:
                        matched = None
                        break
            else:
                matched = best_raw

        results.append({
            "poss_disease": poss,
            "matched_fao_disease": matched,
            "score": round(score, 1)
        })

    mapping_df = pd.DataFrame(results)

    fao_to_poss = {
        row["matched_fao_disease"]: row["poss_disease"]
        for _, row in mapping_df.dropna(subset=["matched_fao_disease"]).iterrows()
    }
    return mapping_df, fao_to_poss

In [79]:
mapping_df, fao_to_poss = map_poss_to_df(fao_reports_df, poss_diseases, threshold=84, one_to_one=False)


In [80]:
mapping_df, fao_to_poss = map_poss_to_df(fao_reports_df, poss_diseases, threshold=84, one_to_one=False)
if 'Newcastle disease' in fao_reports_df.columns:
    fao_to_poss.pop('Newcastle disease')#Already set Newcastle disease (velogenic) in FAO_dict_replace in prior step

fao_reports_df["Disease"] = fao_reports_df["Disease"].map(fao_to_poss)


#Applying rest of FAO report dictionary conversion here
fao_reports_df["Disease"] = fao_reports_df["Disease"].apply(lambda x: FAO_dict_replace.get(x, [x]))
# Explode the column to duplicate rows where needed
fao_reports_df = fao_reports_df.explode("Disease", ignore_index=True)

#Accounting for FAO convention of listing Avian influenza categories by serotype
def map_serotype(row):
    if row["Disease"] == "Influenza - Avian":
        sero = str(row["Serotype"]).upper()
        targets = ["Influenza A virus (Inf. with)"]  # always include general category

        if "HPAI" in sero:
            targets += [
                "High pathogenicity avian influenza viruses (poultry) (Inf. with)",
                "Influenza A viruses of high pathogenicity (Inf. with) (non-poultry including wild birds) (2017-)"
            ]
        elif "LPAI" in sero:
            targets += [
                "Low pathogenic avian influenza (poultry) (2006-2021)",
                "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)"
            ]

        return targets
    return [row["Disease"]]

# Apply to dataframe and duplicate rows where multiple categories apply
fao_reports_df = (
    fao_reports_df
    .assign(Disease=fao_reports_df.apply(map_serotype, axis=1))
    .explode("Disease")
    .reset_index(drop=True)
)

fao_reports_df['Status']='Present'

fao_reports_df_domestic=fao_reports_df[fao_reports_df['Animal.type']=='Domestic']
fao_reports_df_wild=fao_reports_df[fao_reports_df['Animal.type']=='Wild']

In [82]:
#Loading in supplementary disease absence data 
sheets=pd.read_excel(os.path.join(source_data_path, 'Literature data','3Tabs_CountriesDiseaseFreeStatus_and_VaccineAvailability_ForImputations.xlsx'),sheet_name=[0,1])
disease_free_announced=pd.concat(sheets.values(),ignore_index=True)
disease_free_announced['Date'] = pd.to_datetime(disease_free_announced['Date'])
disease_free_announced['Date From'] = pd.to_datetime(disease_free_announced['Date From'])

disease_free_announced=disease_free_announced.loc[:,['Country','Disease','Date From','Date','Source']]
disease_free_announced['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in disease_free_announced['Country']]

# Duplicate rows with "Date From", set Date = Date From (want to list each of these records as two datapoints, one for the year
# it was initially announced as 'disease free', and one for the time the viewed report was made that it is currently disease-free
duplicates = disease_free_announced.loc[disease_free_announced["Date From"].notna()].assign(
    Date=lambda d: d["Date From"]
)

disease_free_announced.loc[disease_free_announced["Date From"].notna(), "Date From"] = np.nan

disease_free_announced = pd.concat([disease_free_announced, duplicates], ignore_index=True)

DF_dict_replace={}

DF_dict_replace['Newcastle disease virus (Inf. with)']=['Newcastle disease (velogenic)'] #In these cases, only velogenic is absent

#Dropping below disease as it is 'fuzzy-matched' to bovine cysticercosis
disease_free_announced=disease_free_announced[disease_free_announced['Disease']!='Porcine cysticercosis']

#Fuzzy matching
mapping_df, disease_free_to_poss = map_poss_to_df(disease_free_announced, poss_diseases, threshold=75, one_to_one=False)

#Other disease alterations
disease_free_announced["Disease"] = disease_free_announced["Disease"].apply(lambda x: DF_dict_replace.get(x, [x]))

# Explode the column to duplicate rows where needed
disease_free_announced = disease_free_announced.explode("Disease", ignore_index=True)
disease_free_announced["Disease"] = (
    disease_free_announced["Disease"]
      .map(disease_free_to_poss)
      .fillna(disease_free_announced["Disease"])   # <- don’t erase on miss
)

#Adding in records that correspond to bTB whenever myc TB is declared free (former is part of latter ccategory)
mask = disease_free_announced["Disease"].eq("Mycobacterium tuberculosis complex (Inf. with)(2019-)")

dupes = disease_free_announced.loc[mask].copy()
dupes["Disease"] = "Bovine tuberculosis (-2018)"

disease_free_announced = pd.concat([disease_free_announced, dupes], ignore_index=True)

disease_free_announced['Status']='Absent'
disease_free_announced['Year']=[i.year for i in disease_free_announced['Date']]

#removing several empty records
disease_free_announced=disease_free_announced[~disease_free_announced['Disease'].isna()]

In [ ]:
combinations = list(itertools.product(poss_ISO3s, poss_years, poss_diseases))

#Making raw dataframe that will fill in
reconciled_present_absent_or_unknown_incidence = pd.DataFrame(combinations, columns=["ISO3", "Year", "Disease"])
reconciled_present_absent_or_unknown_incidence["Region"] = (
    reconciled_present_absent_or_unknown_incidence["ISO3"].map(iso3_region)
)
reconciled_present_absent_or_unknown_incidence['Any cases in country suspected (wildlife included)']=np.nan
reconciled_present_absent_or_unknown_incidence['Source']=np.nan
reconciled_present_absent_or_unknown_incidence['Year data'] = pd.Series([pd.NA] * len(df), dtype='Int64')

reconciled_present_absent_or_unknown_incidence['Any cases in country suspected (wildlife included)']=reconciled_present_absent_or_unknown_incidence['Any cases in country suspected (wildlife included)'].astype("string")
reconciled_present_absent_or_unknown_incidence['Source']=reconciled_present_absent_or_unknown_incidence['Source'].astype("string")

In [ ]:
reconciled_present_absent_or_unknown_incidence

,ISO3,Year,Disease,Region,Any cases in country suspected (wildlife included),Source,Year data
0,ABW,2005,African swine fever virus (Inf. with),Caribbean,<NA>,<NA>,<NA>
1,ABW,2005,Anthrax,Caribbean,<NA>,<NA>,<NA>
2,ABW,2005,Atrophic rhinitis of swine (-2005),Caribbean,<NA>,<NA>,<NA>
3,ABW,2005,Aujeszky's disease virus (Inf. with),Caribbean,<NA>,<NA>,<NA>
4,ABW,2005,Avian chlamydiosis,Caribbean,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...
497947,FLK,2025,Tularemia,Southern Latin America,<NA>,<NA>,<NA>
497948,FLK,2025,Turkey rhinotracheitis (2006-),Southern Latin America,<NA>,<NA>,<NA>
497949,FLK,2025,Venezuelan equine encephalomyelitis,Southern Latin America,<NA>,<NA>,<NA>
497950,FLK,2025,Vesicular stomatitis (-2014),Southern Latin America,<NA>,<NA>,<NA>


In [ ]:
#Assigning here that if there is missing data for "Mycobacterium tuberculosis complex (Inf. with)(2019-)", next look at the other
    #diseases listed for any available information ("Mycobacterium tuberculosis complex (Inf. with)(2019-)" is parent category)
    #i.e., 'aliases'
DISEASE_PRIORITIES = {
    "Mycobacterium tuberculosis complex (Inf. with)(2019-)": [
        "Mycobacterium tuberculosis complex (Inf. with)(2019-)",
        "Bovine tuberculosis (-2018)",
        "Mycobacterium tuberculosis (Inf. with)(-2017)",
    ],
    "Bovine tuberculosis (-2018)": [
        "Bovine tuberculosis (-2018)",
    ],
    "Mycobacterium tuberculosis (Inf. with)(-2017)": [
        "Mycobacterium tuberculosis (Inf. with)(-2017)",
    ],

    # LPAI naming split (virtually a rename across years, as most LPAI serotypes are human-transmissible)
    "Low pathogenic avian influenza (poultry) (2006-2021)": [
        "Low pathogenic avian influenza (poultry) (2006-2021)",
        "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)",
    ],
    "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)": [
        "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)",
        "Low pathogenic avian influenza (poultry) (2006-2021)",
    ],
    
    
    
    
}

def _alias_list(d):
    return DISEASE_PRIORITIES.get(d, [d])

status_precedence = ["Present", "Suspected", "Absent", "Unknown"]

any_suspected_col = "Any cases in country suspected (wildlife included)"

def norm_status(s):
    if s==s:
        return s.capitalize()
    else:
        return "Unknown"

def _best_status(series_like):
    """Pick a single status using precedence: Present > Suspected > Absent > Unknown."""
    vals = [norm_status(x) for x in pd.Series(series_like).dropna()]
    for want in status_precedence:
        if want in vals: return want
    return "Unknown"

def _coerce_year_int64(x):
    if x is None or "Year" not in x.columns: 
        return x
    return x.copy().assign(Year=pd.to_numeric(x["Year"], errors="coerce").astype("Int64"))

def consolidate_status(df, keys=("ISO3","Disease","Year"), status_col="Status", keep_cols=()):
    """
    Collapse duplicates to one row per key:
    - Choose status by precedence Present > Suspected > Absent > Unknown
    - For keep_cols, keep the first non-null (if any), else NaN.
    Returns a dict mapping key -> {'Status':..., <extras>...}
    """
    if df is None or len(df)==0:
        return {}
    use = df.copy()
    use["Status"] = use["Status"].map(norm_status)
    gb = use.groupby(list(keys), dropna=False)

    rows = {}
    for k, sub in gb:
        pick = _best_status(sub[status_col])
        record = {"Status": pick}
        for c in keep_cols:
            first_val = sub[c].dropna().iloc[0] if c in sub and sub[c].notna().any() else np.nan
            record[c] = first_val
        rows[k] = record
    return rows

def has_any_data_keys(*dfs):
    """
    For step #7: 'no data between the 6 dataframes' means none of the six has *any* record for the key,
    regardless of the status value.
    """
    keyset = set()
    for d in dfs:
        if d is not None and len(d):
            keyset |= set(zip(d["ISO3"], d["Disease"], d["Year"]))
    return keyset

def _get_status(m, key):
    return m.get(key, {}).get("Status", "Unknown")

def _get_source(m, key):
    return m.get(key, {}).get("Source", np.nan)

def _first_alias_hit(map_dict, iso, disease, year, accept_set):
    """
    Scan aliases in priority order and return (alias_disease, status) for the first alias
    whose status is in accept_set. If none, return (None, None).
    """
    for d_alias in _alias_list(disease):
        st = _get_status(map_dict, (iso, d_alias, year))
        if st in accept_set:
            return d_alias, st
    return None, None

def _any_alias_hit(map_dict, iso, disease, year, accept_set):
    a, s = _first_alias_hit(map_dict, iso, disease, year, accept_set)
    return a is not None

def reconcile_status(
    reconciled_present_absent_or_unknown_incidence: pd.DataFrame,
    wahis_reported_cases: pd.DataFrame,
    resolved_WAHIS_dis_sit_domestic: pd.DataFrame,
    FAO_reports_domest: pd.DataFrame,
    disease_free_announced: pd.DataFrame,
    resolved_WAHIS_dis_sit_wild: pd.DataFrame,
    FAO_reports_wild: pd.DataFrame,
):
    
    look_back_saved={}
    
    df = reconciled_present_absent_or_unknown_incidence.copy()

    # Just quick check that all exist
    for col in ["ISO3","Disease","Year","Region","Status","Source","Year data", any_suspected_col]:
        if col not in df.columns:
            df[col] = pd.NA  # safer than np.nan for nullable dtypes
    (df,
     wahis_reported_cases,
     resolved_WAHIS_dis_sit_domestic,
     FAO_reports_domest,
     disease_free_announced,
     resolved_WAHIS_dis_sit_wild,
     FAO_reports_wild) = map(_coerce_year_int64, (
        df,
        wahis_reported_cases,
        resolved_WAHIS_dis_sit_domestic,
        FAO_reports_domest,
        disease_free_announced,
        resolved_WAHIS_dis_sit_wild,
        FAO_reports_wild
    ))


    # Consolidate sources
    w_cases_map = consolidate_status(wahis_reported_cases, keep_cols=())
    fao_dom_map = consolidate_status(FAO_reports_domest, keep_cols=())
    res_dom_map = consolidate_status(resolved_WAHIS_dis_sit_domestic, keep_cols=())
    dfa_map     = consolidate_status(disease_free_announced, keep_cols=("Source",))  # keep Status + Source
    res_wild_map= consolidate_status(resolved_WAHIS_dis_sit_wild, keep_cols=())
    fao_wild_map= consolidate_status(FAO_reports_wild, keep_cols=())

    # Check all sources for data
    any_data_keys = has_any_data_keys(
        wahis_reported_cases, resolved_WAHIS_dis_sit_domestic, FAO_reports_domest,
        disease_free_announced, resolved_WAHIS_dis_sit_wild, FAO_reports_wild
    )

    # Default disease statusx = "Unknown"
    df["Status"] = "Unknown"

    # Sort
    df.sort_values(["ISO3","Disease","Year"], inplace=True)
    df_idx = df.set_index(["ISO3","Disease","Year"])
    
    #The algorithm
    for (iso, dis), sub_years in df.groupby(["ISO3","Disease"])["Year"]:
        years_iter = poss_years  
        for y in years_iter:
            key = (iso, dis, y)
            if key not in df_idx.index:
                continue

            cur_status = norm_status(df_idx.at[key, "Status"]) if "Status" in df_idx.columns else "Unknown"
            decided = False

            # Step 1: wahis_reported_cases Present
            if not decided:
                a, st = _first_alias_hit(w_cases_map, iso, dis, y, {"Present"})
                if a:
                    df_idx.at[key, "Status"] = "Present"
                    df_idx.at[key, "Source"] = "WAHIS administrative division report"
                    df_idx.at[key, "Year data"] = y
                    decided = True
                    look_back_saved[(iso,dis)]=('Present','WAHIS administrative division report',y)

            # Step 2: Checking EMPRES-i+ report of presence in domestic
            if not decided:
                a, st = _first_alias_hit(fao_dom_map, iso, dis, y, {"Present"})
                if a:
                    df_idx.at[key, "Status"] = "Present"
                    df_idx.at[key, "Source"] = "EMPRES-i+ report of presence in domestic animals"
                    df_idx.at[key, "Year data"] = y
                    look_back_saved[(iso,dis)]=('Present','EMPRES-i+ report of presence in domestic animals',y)
                    decided = True
                    # Checking if disease is present in country, even if not in domestic animals
                    if _any_alias_hit(fao_wild_map, iso, dis, y, {"Present"}):
                        df_idx.at[key, any_suspected_col] = "Yes"

            # Step 3: Checking WAHIS 'disease situation reports' for domestic animals
            if not decided:
                # Present / Suspected first
                a, st = _first_alias_hit(res_dom_map, iso, dis, y, {"Present","Suspected"})
                if a:
                    df_idx.at[key, "Status"] = "Present"
                    df_idx.at[key, "Source"] = "WAHIS report of presence in domestic animals"
                    df_idx.at[key, "Year data"] = y
                    look_back_saved[(iso,dis)]=('Present','WAHIS report of presence in domestic animals',y)
                    # Checking if disease is present in country, even if not in domestic animals
                    if _any_alias_hit(res_wild_map, iso, dis, y, {"Present","Suspected"}):
                        df_idx.at[key, any_suspected_col] = "Yes"
                    decided = True
                else:
                    # Checking if WAHIS reports absence in domestic
                    a2, st2 = _first_alias_hit(res_dom_map, iso, dis, y, {"Absent"})
                    if a2:
                        df_idx.at[key, "Status"] = "Absent"
                        df_idx.at[key, "Source"] = "WAHIS report of disease absence"
                        df_idx.at[key, "Year data"] = y
                        look_back_saved[(iso,dis)]=('Absent','WAHIS report of disease absence',y)
                        if _any_alias_hit(res_wild_map, iso, dis, y, {"Present","Suspected"}):
                            df_idx.at[key, any_suspected_col] = "Yes"
                        decided = True

            # Step 4: If steps 1-3 did not retrieve anything, checking if supplementary 'disease-free' reports can apply
            if not decided:
                a, st = _first_alias_hit(dfa_map, iso, dis, y, {"Absent"})
                
                #If declared absent, list as absent
                if a:
                    
                    #But if last available report was already 'absent, AS PER WAHIS then use that and its associated source 
                        #(It's possible that the declaration is based on an earlier report from same time as WAHIS-filed report)
                    if (iso,dis) in look_back_saved:
                        look_status,look_source,look_year = look_back_saved[(iso,dis)]
                        if (look_status == 'Absent') & (look_source=='WAHIS report of disease absence'):
                            df_idx.at[key, "Status"] = look_status
                            df_idx.at[key, "Source"] = look_source
                            df_idx.at[key, "Year data"] = look_year
                            
                        else:
                            df_idx.at[key, "Status"] = "Absent"
                            df_idx.at[key, "Source"] = 'Declared disease-free; '+_get_source(dfa_map, (iso, a, y))
                            df_idx.at[key, "Year data"] = y
                            look_back_saved[(iso,dis)]=('Absent','Declared disease-free; '+_get_source(dfa_map, (iso, a, y)), y)
                    
                    else:
                        df_idx.at[key, "Status"] = "Absent"
                        df_idx.at[key, "Source"] = 'Declared disease-free; '+ _get_source(dfa_map, (iso, a, y))
                        df_idx.at[key, "Year data"] = y
                        look_back_saved[(iso,dis)]=('Absent','Declared disease-free; '+_get_source(dfa_map, (iso, a, y)), y)
                    
                    decided = True
                    
            # Step 5: If still unknown about the domestic situation, check if there are EMPRES-i+ reports in wild animals (for inference)
            if not decided:
                a, st = _first_alias_hit(fao_wild_map, iso, dis, y, {"Present"})
                if a:
                    df_idx.at[key, "Status"] = "Present"
                    df_idx.at[key, "Source"] = 'Inferred present - EMRPRES-i+ report of presence in wild animals'
                    df_idx.at[key, "Year data"] = y
                    decided = True
                    look_back_saved[(iso,dis)]=('Present','Inferred present - EMRPRES-i+ report of presence in wild animals',y)

            # Step 6: If still unknown about the domestic situation, check if there are WAHIS reports in wild animals (for inference)
            if not decided:
                a, st = _first_alias_hit(res_wild_map, iso, dis, y, {"Present","Suspected"})
                if a:
                    df_idx.at[key, "Status"] = "Present"
                    df_idx.at[key, "Source"] = "Inferred present - WAHIS report of presence in wild animals"
                    df_idx.at[key, "Year data"] = y
                    decided = True
                    look_back_saved[(iso,dis)]=('Present','Inferred present - WAHIS report of presence in wild animals',y)

            # Step 7: if no information thus far for this year, look back at last available information gathered from one of above sources for this country-disease pair
            if not decided:
                if (iso,dis) in look_back_saved:
                    look_status,look_source,look_year = look_back_saved[(iso,dis)]
                    df_idx.at[key, "Status"] = look_status
                    df_idx.at[key, "Source"] = look_source
                    df_idx.at[key, "Year data"] = look_year
                    decided = True
            # else if still Unknown; final regional step will handle this

    df = df_idx.reset_index()

    # Final step (regional majority for remaining Unknowns) 
    known = df[df["Status"].isin(["Present","Absent"])]

    counts = (
        known
        .groupby(["Region","Disease","Year","Status"])
        .size()
        .unstack("Status")
        .fillna(0)
    )

    def _majority_for(region, disease, year):
        # sum Present/Absent over for this disease
        diseases_to_count = _alias_list(disease)
        p = a = 0
        for dd in diseases_to_count:
            try:
                row = counts.loc[(region, dd, year)]
                p += int(row.get("Present", 0))
                a += int(row.get("Absent", 0))
            except KeyError:
                pass
        if p==0 and a==0: return "NONE"
        if p > a: return "Present"
        if a > p: return "Absent"
        return "TIE"

    unk_mask = df["Status"].eq("Unknown")
    for idx, r in df[unk_mask].iterrows():
        region, disease, year, iso = r["Region"], r["Disease"], r["Year"], r["ISO3"]
        maj = _majority_for(region, disease, year)

        if maj == "Present":
            df.at[idx, "Status"] = "Present"
            df.at[idx, "Source"] = "Inferred present - present in the majority of reporting countries in the same Global Health Data Exchange region"
            df.at[idx, "Year data"] = year
        elif maj == "Absent":
            df.at[idx, "Status"] = "Absent"
            df.at[idx, "Source"] = "Inferred disease-free - majority of reporting countries in the same Global Health Data Exchange region are disease-free"
            df.at[idx, "Year data"] = year
        elif maj == "TIE":
            # If majority voting fails for domestic animals (insufficient data), last check to see if there is disease-free status reported in wild animals
            key = (iso, disease, year)
            st_wild = _get_status(res_wild_map, key)
            if st_wild in {"Absent", "Unknown"}:
                df.at[idx, "Status"] = st_wild
                if st_wild == "Absent":
                    df.at[idx, "Source"] = "Inferred disease-free - WAHIS report of absence in wild animals"
                    df.at[idx, "Year data"] = year

    # Noting here if disease exists in the country at all (not in reference to domestic animals)
    df.loc[df["Status"].eq("Present"), any_suspected_col] = "Yes"
    df.loc[df[any_suspected_col].isna() & df["Status"].eq("Absent"), any_suspected_col] = "No"

    for c in ["Status","Source", any_suspected_col]:
        if c in df.columns:
            df[c] = df[c].astype("string")

    return df


In [86]:
reconciled_updated = reconcile_status(
reconciled_present_absent_or_unknown_incidence,
wahis_reported_cases,
resolved_WAHIS_dis_sit_domestic,
fao_reports_df_domestic,
disease_free_announced,
resolved_WAHIS_dis_sit_wild,
fao_reports_df_wild)

In [90]:
# "Influenza A virus (Inf. with)" is a separate category collected by WAHIS. Here we ensure that if any influenza is noted present, "Influenza A virus (Inf. with)" is as well
variants = [
    'High pathogenicity avian influenza viruses (poultry) (Inf. with)',
    'Influenza A viruses of high pathogenicity (Inf. with) (non-poultry including wild birds) (2017-)',
    'Low pathogenic avian influenza (poultry) (2006-2021)',
    'Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)'
]

df = reconciled_updated

# 1) Collect Source 
donors = (
    df.loc[df['Disease'].isin(variants) & (df['Status'] == 'Present'),
           ['ISO3', 'Year', 'Source', 'Year data']]
      .groupby(['ISO3', 'Year'], as_index=False)
      .agg(
          Source_variant=('Source', lambda s: '; '.join(sorted({str(x) for x in s.dropna()}))),
          Year_data=('Year data', 'first')   # or 'max' / 'min' / 'unique' / custom join
      )
)


# 2) Influenza A rows that are NOT 'Present'
mask_inflA_not_present = (
    (df['Disease'] == 'Influenza A virus (Inf. with)') &
    (df['Status'] != 'Present')
)

# 3) Align those rows with donor matches on (ISO3, Year)
receivers = df.loc[mask_inflA_not_present, ['ISO3', 'Year']].merge(
    donors, on=['ISO3', 'Year'], how='left'
)

# 4) Update only where a donor 'Present' exists
has_donor = receivers['Source_variant'].notna().values
idx_to_update = df.index[mask_inflA_not_present][has_donor]

df.loc[idx_to_update, 'Status'] = 'Present'
df.loc[idx_to_update, 'Any cases in country suspected (wildlife included)'] = 'Yes'

df.loc[idx_to_update, 'Source'] = (
    'Reconciled with another influenza category: ' +
    receivers.loc[has_donor, 'Source_variant'].astype(str).values
)
df.loc[idx_to_update, 'Year data']=receivers.loc[has_donor, 'Year_data'].values


In [92]:
reconciled_updated.rename(columns={'Status':'Status in livestock'},inplace=True)
reconciled_updated.to_csv(os.path.join(notebook_dir,'Common Source Data','Processed data','Multisource','domestic_disease_presence_or_absence_filter.csv'),
                         index=False)